# Import Libraries

In [1]:
# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# API
import requests
import json

# Progress bar
from tqdm import tqdm

# Paths
import os
data_path = os.path.join('..', 'data')

# Prepare Data

## Load Dataset

In [2]:
# Read csv
csv_path = os.path.join(data_path, 'genes_human.csv')
csv_dataframe = pd.read_csv(csv_path, sep='\t')

## View Data

In [5]:
df = csv_dataframe.copy()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80225 entries, 0 to 80224
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   80225 non-null  int64  
 1   seq          80225 non-null  object 
 2   gene         80225 non-null  object 
 3   chr          80225 non-null  object 
 4   strand       80225 non-null  object 
 5   coord_start  80225 non-null  int64  
 6   coord_end    80225 non-null  int64  
 7   canonical    80225 non-null  bool   
 8   ENC          80225 non-null  float64
 9   transcript   80225 non-null  object 
 10  exp_T        80225 non-null  float64
 11  exp_HEK293   80225 non-null  float64
 12  exp_U2OS     80225 non-null  float64
 13  cARS_T       80225 non-null  float64
 14  cARS_HEK293  80225 non-null  float64
 15  cARS_U2OS    80225 non-null  float64
dtypes: bool(1), float64(7), int64(3), object(5)
memory usage: 9.3+ MB


In [7]:
df.head()

,Unnamed: 0,seq,gene,chr,strand,coord_start,coord_end,canonical,ENC,transcript,exp_T,exp_HEK293,exp_U2OS,cARS_T,cARS_HEK293,cARS_U2OS
0,0,ATGGCGTCCCCGTCTCGGAGACTGCAGACTAAACCAGTCATTACTT...,ENSG00000000003,X,-,100627107,100636806,True,45.363981,ENST00000373020,6.113815,9.838259,14.0,0.0,0.0,0.0
1,1,ATGCTAAAACTGTATGCAATGTTTCTGACTCTCGTTTTTTTGGTCG...,ENSG00000000003,X,-,100627108,100637104,False,38.175250,ENST00000612152,6.113815,9.838259,14.0,0.0,0.0,0.0
2,2,ATGGCAAAGAATCCTCCAGAGAATTGTGAAGACTGTCACATTCTAA...,ENSG00000000005,X,+,100584935,100599885,True,45.428790,ENST00000373031,3.349926,2.617150,0.0,0.0,0.0,0.0
3,3,ATGGCCTCCTTGGAAGTCAGTCGTAGTCCTCGCAGGTCTCGGCGGG...,ENSG00000000419,20,-,50934866,50958550,False,46.773340,ENST00000466152,11.399582,10.070030,213.0,0.0,0.0,0.0
4,4,ATGGCCTCCTTGGAAGTCAGTCGTAGTCCTCGCAGGTCTCGGCGGG...,ENSG00000000419,20,-,50934866,50958555,False,47.072886,ENST00000371582,11.399582,10.070030,213.0,0.0,0.0,0.0


# Get Genotype Tissue Expression (GTEx) Vectors

## Util Functions

In [8]:
def filter_by_transcripts(data, transcript_patterns):
  """Get 'data' list of dictionaries. 
  Filter data by transcript patterns."""

  transcript_key = 'transcriptId'

  filtered = [d for d in data if (transcript_key in d.keys()) and (d[transcript_key].split('.')[0] in transcript_patterns)]
  
  return filtered


def get_tissue_site_details():
    """Retrieve a list of all tissueSiteDetailId values from the GTEx API."""
    url = 'https://gtexportal.org/api/v2/dataset/tissueSiteDetail'
    response = requests.get(url)
    
    if response.status_code == 200:
        data = json.loads(response.text)
        tissue_site_details = [item['tissueSiteDetailId'] for item in data['data']]
        return tissue_site_details
    else:
        print(f"Failed to retrieve data: {response.status_code}")
        return []


def retrieve_median_transcript_expressions(gene_ids, transcript_ids, suffix_range=15, batch_size=82, range_start=1):
  """Get gene IDs without versions, find the gencodeID versions, and get all 
  median transcript expressions from GTEx Portal API for a list of gene ids.
  
  Args:
    gene_ids (list): List of gene ids without version suffixes.
    suffix_range (int): The search range of version suffixes [1,...,suffix_range].
    
  Returns:
    data (list): List of dictionaries containing the median transcript expression data.
  """

  # 'batch_size' too big will cause a '413 ERROR' 
  num_batches = np.ceil(len(gene_ids) / batch_size).astype(int)

  data = []

  for i in tqdm(range(num_batches), desc='Retrieving data from GTEx Portal API'):
    start = i * batch_size
    end = min((i + 1) * batch_size, len(gene_ids))

    # Get batch of gene ids
    gene_batch = gene_ids[start:end]
    transcript_batch = transcript_ids[start:end]
    
    # Retrieve data from GTEx Portal API
    suffixes = range(range_start, range_start+suffix_range)
    all_gencodes_batch = [f"{gene_id}.{suffix}" for gene_id in gene_batch for suffix in suffixes]
    url = 'https://gtexportal.org/api/v2/expression/medianTranscriptExpression'
    params = {"gencodeId": all_gencodes_batch, 'itemsPerPage': 100000}

    response = requests.get(url, params=params)
    
    # Check if response is successful
    if response.status_code != 200:
      print(f"{response}. Failed to retrieve data for {gene_batch}")
      continue
    
    batch_data = json.loads(response.text)["data"]

    # Filter data by transcript ids
    filtered_batch_data = filter_by_transcripts(batch_data, transcript_batch)

    data += filtered_batch_data

  return data


def get_median_transcript_gtex(gene_ids, transcript_ids, suffix_range=15, batch_size=82, range_start=1):
  # retrieve data from GTEx Portal API
  data = retrieve_median_transcript_expressions(gene_ids, transcript_ids, suffix_range, batch_size, range_start)

  if not data:
    return -1
  
  # Group by transcript id
  data_df = pd.DataFrame(data)[['transcriptId', 'median', 'tissueSiteDetailId']]
  grouped_data_df = data_df.groupby('transcriptId')

  # Aggregate values into lists
  aggregated_df = grouped_data_df.agg(lambda x: list(x)).reset_index()

  # Remove version suffixes
  aggregated_df['transcriptId'] = aggregated_df['transcriptId'].str[:15]#.tolist()

  # Original transcript id order
  order_df = pd.DataFrame({'transcriptId': transcript_ids})

  # Reorder aggregated data
  merged_df = order_df.merge(aggregated_df, on='transcriptId', how='left')

  # Rename transcriptId column for compatibility with CSV data
  merged_df.rename(columns={'transcriptId': 'transcript'}, inplace=True)

  return merged_df  # , _tissues, _median_transcript_gtex

## Example Vector

In [9]:
random_row = df.loc[np.random.randint(0, len(df))]

gene_id = [random_row['gene']]
transcript_id = [random_row['transcript']]

data = get_median_transcript_gtex(gene_id, transcript_id)
pd.DataFrame(data)

Retrieving data from GTEx Portal API: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]


,transcript,median,tissueSiteDetailId
0,ENST00000414305,"[0.009999999776482582, 0.0, 0.1500000059604644...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."


In [10]:
random_row

Unnamed: 0                                                 45982
seq            ATGGAATCTACGTTGACTTTAGCAACGGAACAACCTGTTAAGAAGA...
gene                                             ENSG00000154269
chr                                                            6
strand                                                         +
coord_start                                            131628460
coord_end                                              131747418
canonical                                                  False
ENC                                                    50.996557
transcript                                       ENST00000414305
exp_T                                                   5.144832
exp_HEK293                                              4.094387
exp_U2OS                                                     0.1
cARS_T                                                       0.0
cARS_HEK293                                                  0.0
cARS_U2OS                

## Compare Vectors

**Compare Different Transcripts of same Genotype**

In [11]:
gene_ids = ["ENSG00000000003", "ENSG00000000003"]
transcript_ids = ["ENST00000373020", "ENST00000612152"]

data = get_median_transcript_gtex(gene_ids, transcript_ids)

a = np.array(data.loc[0, 'median'])
b = np.array(data.loc[1, 'median'])

print(f"\nCosine Angle: {a @ b / (np.linalg.norm(a) * np.linalg.norm(b))}")

Retrieving data from GTEx Portal API: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]


Cosine Angle: 0.9127356131192652


**Look at a well known gene (HBA1):**

In [12]:
df.loc[df['gene'] == "ENSG00000206172"]

,Unnamed: 0,seq,gene,chr,strand,coord_start,coord_end,canonical,ENC,transcript,exp_T,exp_HEK293,exp_U2OS,cARS_T,cARS_HEK293,cARS_U2OS
74822,74822,ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGG...,ENSG00000206172,16,+,176679,177522,True,30.150536,ENST00000320868,0.0,0.0,0.0,0.0,0.0,0.0
74823,74823,ATGTTCCTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCG...,ENSG00000206172,16,+,176703,177522,False,27.574955,ENST00000397797,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
# HBA1 transcripts
gene_ids = ["ENSG00000206172", "ENSG00000206172"]
transcript_ids = ["ENST00000320868", "ENST00000397797"]

data = get_median_transcript_gtex(gene_ids, transcript_ids)

med_exp_1 = np.array(data.loc[0, 'median'])
tissues_1 = np.array(data.loc[0, 'tissueSiteDetailId'])

med_exp_2 = np.array(data.loc[1, 'median'])
tissues_2 = np.array(data.loc[1, 'tissueSiteDetailId'])

max_exp_1 = np.argsort(med_exp_1)[::-1]
max_exp_2 = np.argsort(med_exp_2)[::-1]

print(f"\nFirst 10 max expressed tissues of HBA1 transcript {transcript_ids[0]}\n\n", np.array(tissues_1)[max_exp_1][:10], np.array(med_exp_1)[max_exp_1][:10])

print(f"\nFirst 10 max expressed tissues of HBA1 transcript {transcript_ids[1]}\n\n", np.array(tissues_2)[max_exp_2][:10], np.array(med_exp_2)[max_exp_2][:10])

print(f"\nCosine Angle: {med_exp_1 @ med_exp_2 / (np.linalg.norm(med_exp_1) * np.linalg.norm(med_exp_2))}")

Retrieving data from GTEx Portal API: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]


First 10 max expressed tissues of HBA1 transcript ENST00000320868

 ['Whole_Blood' 'Spleen' 'Lung' 'Kidney_Medulla' 'Kidney_Cortex'
 'Adipose_Visceral_Omentum' 'Pituitary' 'Brain_Spinal_cord_cervical_c-1'
 'Brain_Substantia_nigra' 'Thyroid'] [125309.15625      2128.26000977    693.36999512    499.76000977
    405.76998901    332.33999634    279.51998901    251.78999329
    241.27999878    222.8999939 ]

First 10 max expressed tissues of HBA1 transcript ENST00000397797

 ['Whole_Blood' 'Spleen' 'Uterus' 'Thyroid' 'Testis' 'Vagina' 'Stomach'
 'Small_Intestine_Terminal_Ileum' 'Skin_Sun_Exposed_Lower_leg'
 'Skin_Not_Sun_Exposed_Suprapubic'] [23.98999977  0.83999997  0.          0.          0.          0.
  0.          0.          0.          0.        ]

Cosine Angle: 0.9997844630185476


**We can see that the first transcript is much more expressed but when normalizing they are close.**

**Compare HBA1 with ASPM:**

In [14]:
print("ASPM:")
df.loc[df['gene']=="ENSG00000066279"]

ASPM:


,Unnamed: 0,seq,gene,chr,strand,coord_start,coord_end,canonical,ENC,transcript,exp_T,exp_HEK293,exp_U2OS,cARS_T,cARS_HEK293,cARS_U2OS
4842,4842,ATGGCGAACCGGCGAGTGGGGCGAGGCTGCTGGGAAGTGAGCCCGA...,ENSG00000066279,1,-,197084126,197146669,True,46.312858,ENST00000367409,4.599783,9.871607,77.0,0.0,0.0,0.0
4843,4843,ATGGCGAACCGGCGAGTGGGGCGAGGCTGCTGGGAAGTGAGCCCGA...,ENSG00000066279,1,-,197084126,197146669,False,46.649441,ENST00000680265,4.599783,9.871607,77.0,0.0,0.0,0.0
4844,4844,ATGGCGAACCGGCGAGTGGGGCGAGGCTGCTGGGAAGTGAGCCCGA...,ENSG00000066279,1,-,197084127,197146694,False,48.385063,ENST00000294732,4.599783,9.871607,77.0,0.0,0.0,0.0
4845,4845,ATGGCGAACCGGCGAGTGGGGCGAGGCTGCTGGGAAGTGAGCCCGA...,ENSG00000066279,1,-,197084323,197146437,False,46.319705,ENST00000680710,4.599783,9.871607,77.0,0.0,0.0,0.0
4846,4846,ATGGCGAACCGGCGAGTGGGGCGAGGCTGCTGGGAAGTGAGCCCGA...,ENSG00000066279,1,-,197104717,197146437,False,50.032129,ENST00000681879,4.599783,9.871607,77.0,0.0,0.0,0.0


In [15]:
# HBA1, ASPM transcripts
gene_ids = ["ENSG00000206172", "ENSG00000066279"]
transcript_ids = ["ENST00000320868", "ENST00000367409"]

data = get_median_transcript_gtex(gene_ids, transcript_ids)

med_exp_1 = np.array(data.loc[0, 'median'])
tissues_1 = np.array(data.loc[0, 'tissueSiteDetailId'])

med_exp_2 = np.array(data.loc[1, 'median'])
tissues_2 = np.array(data.loc[1, 'tissueSiteDetailId'])

max_exp_1 = np.argsort(med_exp_1)[::-1]
max_exp_2 = np.argsort(med_exp_2)[::-1]

print(f"\nFirst 10 max expressed tissues of HBA1 transcript {transcript_ids[0]}\n\n", np.array(tissues_1)[max_exp_1][:10], np.array(med_exp_1)[max_exp_1][:10])

print(f"\nFirst 10 max expressed tissues of ASPM transcript {transcript_ids[1]}\n\n", np.array(tissues_2)[max_exp_2][:10], np.array(med_exp_2)[max_exp_2][:10])

print(f"\nCosine Angle: {med_exp_1 @ med_exp_2 / (np.linalg.norm(med_exp_1) * np.linalg.norm(med_exp_2))}")

Retrieving data from GTEx Portal API: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s]


First 10 max expressed tissues of HBA1 transcript ENST00000320868

 ['Whole_Blood' 'Spleen' 'Lung' 'Kidney_Medulla' 'Kidney_Cortex'
 'Adipose_Visceral_Omentum' 'Pituitary' 'Brain_Spinal_cord_cervical_c-1'
 'Brain_Substantia_nigra' 'Thyroid'] [125309.15625      2128.26000977    693.36999512    499.76000977
    405.76998901    332.33999634    279.51998901    251.78999329
    241.27999878    222.8999939 ]


IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [ ]:
# HBA1 transcripts
gene_ids = ["ENSG00000206172", "ENSG00000066279"]
transcript_ids = ["ENST00000320868", "ENST00000367409"]

data = get_median_transcript_gtex(gene_ids, transcript_ids)

med_exp_1 = np.array(data.loc[0, 'median'])
tissues_1 = np.array(data.loc[0, 'tissueSiteDetailId'])

med_exp_2 = np.array(data.loc[1, 'median'])
tissues_2 = np.array(data.loc[1, 'tissueSiteDetailId'])

max_exp_1 = np.argsort(med_exp_1)[::-1]
max_exp_2 = np.argsort(med_exp_2)[::-1]

print(f"\nFirst 10 max expressed tissues of HBA1 transcript {transcript_ids[0]}\n\n", np.array(tissues_1)[max_exp_1][:10], np.array(med_exp_1)[max_exp_1][:10])

print(f"\nFirst 10 max expressed tissues of HBA1 transcript {transcript_ids[1]}\n\n", np.array(tissues_2)[max_exp_2][:10], np.array(med_exp_2)[max_exp_2][:10])

print(f"\nCosine Angle: {med_exp_1 @ med_exp_2 / (np.linalg.norm(med_exp_1) * np.linalg.norm(med_exp_2))}")

Retrieving data from GTEx Portal API: 100%|██████████| 1/1 [00:00<00:00,  2.97it/s]


First 10 max expressed tissues of HBA1 transcript ENST00000320868

 ['Whole_Blood' 'Spleen' 'Lung' 'Kidney_Medulla' 'Kidney_Cortex'
 'Adipose_Visceral_Omentum' 'Pituitary' 'Brain_Spinal_cord_cervical_c-1'
 'Brain_Substantia_nigra' 'Thyroid'] [125309.15625      2128.26000977    693.36999512    499.76000977
    405.76998901    332.33999634    279.51998901    251.78999329
    241.27999878    222.8999939 ]

First 10 max expressed tissues of HBA1 transcript ENST00000367409

 ['Cells_EBV-transformed_lymphocytes' 'Cells_Cultured_fibroblasts' 'Testis'
 'Esophagus_Mucosa' 'Skin_Sun_Exposed_Lower_leg'
 'Skin_Not_Sun_Exposed_Suprapubic' 'Small_Intestine_Terminal_Ileum'
 'Vagina' 'Spleen' 'Colon_Transverse'] [8.31999969 3.625      1.61000001 1.03999996 0.49000001 0.47
 0.34       0.31999999 0.27000001 0.23      ]

Cosine Angle: 0.002097024198917175


# Add Expression Data Column to Dataframe

In [14]:
gene_ids = df['gene'].tolist()
transcript_ids = df['transcript'].tolist()

data = get_median_transcript_gtex(gene_ids, transcript_ids)

merged_df = df.merge(data, on="transcript", how="left")

Retrieving data from GTEx Portal API: 100%|██████████| 979/979 [18:46<00:00,  1.15s/it]


In [19]:
merged_df

,seq,gene,chr,strand,coord_start,coord_end,canonical,ENC,transcript,exp_T,exp_HEK293,exp_U2OS,cARS_T,cARS_HEK293,cARS_U2OS,median,tissueSiteDetailId
0,ATGGCGTCCCCGTCTCGGAGACTGCAGACTAAACCAGTCATTACTT...,ENSG00000000003,X,-,100627107,100636806,True,45.363981,ENST00000373020,6.113815,9.838259,14.0,0.0,0.0,0.0,"[25.280000686645508, 22.770000457763672, 15.26...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
1,ATGCTAAAACTGTATGCAATGTTTCTGACTCTCGTTTTTTTGGTCG...,ENSG00000000003,X,-,100627108,100637104,False,38.175250,ENST00000612152,6.113815,9.838259,14.0,0.0,0.0,0.0,"[1.0399999618530273, 0.8700000047683716, 0.485...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
2,ATGGCAAAGAATCCTCCAGAGAATTGTGAAGACTGTCACATTCTAA...,ENSG00000000005,X,+,100584935,100599885,True,45.428790,ENST00000373031,3.349926,2.617150,0.0,0.0,0.0,0.0,"[15.739999771118164, 6.639999866485596, 0.0, 0...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
3,ATGGCCTCCTTGGAAGTCAGTCGTAGTCCTCGCAGGTCTCGGCGGG...,ENSG00000000419,20,-,50934866,50958550,False,46.773340,ENST00000466152,11.399582,10.070030,213.0,0.0,0.0,0.0,"[1.0700000524520874, 0.9200000166893005, 0.699...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
4,ATGGCCTCCTTGGAAGTCAGTCGTAGTCCTCGCAGGTCTCGGCGGG...,ENSG00000000419,20,-,50934866,50958555,False,47.072886,ENST00000371582,11.399582,10.070030,213.0,0.0,0.0,0.0,"[1.3700000047683716, 1.600000023841858, 1.5449...","[Adipose_Subcutaneous, Adipose_Visceral_Omentu..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80220,ATGATTCCCTGCAGTAAACGGACTTTTCATTTATTTAATCATTCAA...,ENSG00000291315,3,+,40312085,40312214,True,26.661368,ENST00000706951,0.000000,0.000000,0.0,0.0,0.0,0.0,NaN,NaN
80221,ATGGCCCCCAAGCCGGGGGCCGAGTGGAGCACAGCCCTGTCCCATC...,ENSG00000291316,8,-,144449581,144465035,True,36.956221,ENST00000438911,0.000000,0.000000,0.0,0.0,0.0,0.0,NaN,NaN
80222,ATGGCCCCCAAGCCGGGGGCCGAGTGGAGCACAGCCCTGTCCCATC...,ENSG00000291317,8,-,144463816,144465489,True,39.257905,ENST00000403000,0.000000,0.000000,0.0,0.0,0.0,0.0,NaN,NaN
80223,ATGGCCCCCAAGCCGGGGGCCGAGTGGAGCACAGCCCTGTCCCATC...,ENSG00000291317,8,-,144463816,144465648,False,39.257905,ENST00000424149,0.000000,0.000000,0.0,0.0,0.0,0.0,NaN,NaN


## Increment suffix search range for remaining Na values:

In [136]:
na_rows = merged_df[merged_df.isna().any(axis=1)]
na_gene_ids = na_rows['gene'].tolist()
na_transcript_ids = na_rows['transcript'].tolist()

na_data = get_median_transcript_gtex(na_gene_ids, na_transcript_ids, suffix_range=18, batch_size=67)

na_df = df.merge(na_data, on="transcript", how="left")

updated_df = merged_df.combine_first(na_df)

Retrieving data from GTEx Portal API: 100%|██████████| 559/559 [09:42<00:00,  1.04s/it]


In [137]:
updated_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 80225 entries, 0 to 80224
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   seq                 80225 non-null  object 
 1   gene                80225 non-null  object 
 2   chr                 80225 non-null  object 
 3   strand              80225 non-null  object 
 4   coord_start         80225 non-null  int64  
 5   coord_end           80225 non-null  int64  
 6   canonical           80225 non-null  bool   
 7   ENC                 80225 non-null  float64
 8   transcript          80225 non-null  object 
 9   exp_T               80225 non-null  float64
 10  exp_HEK293          80225 non-null  float64
 11  exp_U2OS            80225 non-null  float64
 12  cARS_T              80225 non-null  float64
 13  cARS_HEK293         80225 non-null  float64
 14  cARS_U2OS           80225 non-null  float64
 15  median              57591 non-null  object 
 16  tiss

## #2 Increase suffix search range for remaining Na values:**

In [ ]:
na_rows_2 = updated_df[updated_df.isna().any(axis=1)]
na_gene_ids_2 = na_rows_2['gene'].tolist()
na_transcript_ids_2 = na_rows_2['transcript'].tolist()

na_data_2 = get_median_transcript_gtex(na_gene_ids_2, na_transcript_ids_2, suffix_range=32, batch_size=35)

na_df_2 = df.merge(na_data_2, on="transcript", how="left")

updated_df_2 = updated_df.combine_first(na_df_2)

In [92]:
updated_df_2.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 80225 entries, 0 to 80224
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   seq                 80225 non-null  object 
 1   gene                80225 non-null  object 
 2   chr                 80225 non-null  object 
 3   strand              80225 non-null  object 
 4   coord_start         80225 non-null  int64  
 5   coord_end           80225 non-null  int64  
 6   canonical           80225 non-null  bool   
 7   ENC                 80225 non-null  float64
 8   transcript          80225 non-null  object 
 9   exp_T               80225 non-null  float64
 10  exp_HEK293          80225 non-null  float64
 11  exp_U2OS            80225 non-null  float64
 12  cARS_T              80225 non-null  float64
 13  cARS_HEK293         80225 non-null  float64
 14  cARS_U2OS           80225 non-null  float64
 15  median              63472 non-null  object 
 16  tiss

## #3 Increase search range for remaining Na values:

In [ ]:
na_rows_3 = updated_df_2[updated_df_2.isna().any(axis=1)]
na_gene_ids_3 = na_rows_3['gene'].tolist()
na_transcript_ids_3 = na_rows_3['transcript'].tolist()

na_data_3 = get_median_transcript_gtex(na_gene_ids_3, na_transcript_ids_3, suffix_range=48, batch_size=23, range_start=51)

In [112]:
na_data_3

-1

## Save Dataframe with Expression Data to CSV:

In [ ]:
# rename 'median' column to 'median_float'
updated_df_2.rename(columns={'median': 'median_float'}, inplace=True)

# save csv to data folder
csv_output_path = os.path.join(data_path, 'genes_human_with_expression.pkl')

updated_df_2.to_pickle(csv_output_path, index=False)

**Check that all Expression Vectors are Length 54:**

In [ ]:
col = updated_df_2['median_float']

all_lengths_54 = all(len(lst) == 54 for lst in updated_df_2['median_float'] if isinstance(lst, list))

all_lengths_54

True

**Ratio of non-Na Valued Rows:**

In [119]:
len(updated_df_2.dropna()) / len(updated_df_2)

0.7911748208164537